In [4]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

import icarogw
import healpy as hp
import numpy as np
import pandas as pd
from  astropy.cosmology import FlatLambdaCDM

In [10]:
#############################################
# GENERAL SETTINGS
#############################################
# Choose nside (the same as in HI maps)
nside = 16
# Choose random seed for reproducitbily
my_random_seed = 1
# Choose whether to use HI maps in GW data 
# generation or not
hi_probability = True

In [11]:
npix = hp.nside2npix(nside)
lmax = 3*nside 

# redhsift range for reference
zmin = 0.005
zmax = 3.025

H0_value = 67.7
# Reference cosmology
cosmo_ref = icarogw.cosmology.astropycosmology(zmax=20.)
cosmo_ref.build_cosmology(FlatLambdaCDM(H0=H0_value,Om0=0.308)) 

In [12]:
PATH_TO_DENSITY_MATRIX = 'hi_data/'

In [13]:
if hi_probability:
    density_matrix = pd.read_hdf(PATH_TO_DENSITY_MATRIX + 'map4icaro_lmax' + str(lmax) + '_zmin' + str(zmin) + '_zmax' + str(zmax) + '_nside' + str(nside) + '.hdf5').to_numpy()
else:
    density_matrix = pd.read_hdf(PATH_TO_DENSITY_MATRIX + 'map4icaro_lmax' + str(lmax) + '_zmin' + str(zmin) + '_zmax' + str(zmax) + '_nside' + str(nside) + '.hdf5').to_numpy()
    density_matrix = np.ones_like(density_matrix)

In [18]:
redshift_grid = np.array([0.005, 0.015, 0.035, 0.06 , 0.085, 0.11 , 0.135, 0.17 , 0.205,
        0.24 , 0.28 , 0.32 , 0.36 , 0.405, 0.455, 0.505, 0.56 , 0.62 ,
        0.68 , 0.745, 0.82 , 0.895, 0.975, 1.06 , 1.155, 1.26 , 1.365,
        1.485, 1.615, 1.755, 1.905, 2.05 , 2.205, 2.39 , 2.585, 2.795,
        3.025])

In [19]:
pixel_grid = np.arange(0, npix, 1).astype(int) # generates an array of (integer) numbers from 0 to npix-1

In [ ]:
HI = icarogw.HI.HI_map(redshift_grid, pixel_grid, density_matrix, bGW=1., alphaGW=0.)

In [ ]:
# Simulate GW Events
Nbinaries = int(1e4) # More or less one year of observation
np.random.seed(my_random_seed)

# This is the redshift evaluation rate model we use
rw = icarogw.wrappers.rateevolution_Madau()
rw.update(gamma=2.7,kappa=6.,zp=1.)

# Draw more binaries before reweighting
z_injection = np.random.uniform(redshift_grid.min(),redshift_grid.max(),size=10*Nbinaries)
pixel_injection = np.random.randint(len(pixel_grid),size=10*Nbinaries)

# Put the CBCs in local overdensities of the map
log_weights = rw.rate.log_evaluate(z_injection) + np.log(cosmo_ref.dVc_by_dzdOmega_at_z(z_injection)) + \
              np.log(HI.drho_dzdomega(z_injection,pixel_injection,cosmo_ref)) - np.log1p(z_injection)
log_weights = log_weights.flatten()
log_weights-=log_weights.max() # for numerical stability, so that the largest weight is 1 
wei = np.exp(log_weights) # taking the exp, we get values between 0 and 1


idx_binaries = np.random.choice(len(z_injection),size=Nbinaries,replace=True,p=wei/wei.sum())
pix_injections = pixel_injection[idx_binaries]
ra_injections, dec_injections = icarogw.conversions.indices2radec(pix_injections, nside, nest=False)
red_injections = z_injection[idx_binaries]
dl_injections = cosmo_ref.z2dl(red_injections)


mass_1 = np.random.uniform(5,300,size=Nbinaries)
mass_2 = np.random.uniform(5,300,size=Nbinaries)
theta_jn = np.arccos(np.random.uniform(-1,1,size=Nbinaries))
polarization = np.random.uniform(0,np.pi,size=Nbinaries)
phase = np.random.uniform(0,2*np.pi,size=Nbinaries)
a_1 = np.random.uniform(0.005,0.99,size=Nbinaries)
a_2 = np.random.uniform(0.005,0.99,size=Nbinaries)
tilt_1 = np.arccos(np.random.uniform(-1,1,size=Nbinaries))
tilt_2 = np.arccos(np.random.uniform(-1,1,size=Nbinaries))
phi_12 = np.random.uniform(0,2*np.pi,size=Nbinaries)
phi_jl = np.random.uniform(0,2*np.pi,size=Nbinaries)
times = np.zeros(Nbinaries) # coalescence time at the geocenter


In [23]:
# create dataframe with events to pass to GWFish for SNR computation
data = {'mass_1': mass_1,
        'mass_2': mass_2,
        'luminosity_distance': dl_injections,
        'theta_jn': theta_jn,
        'ra': ra_injections,
        'dec': dec_injections,
        'psi': polarization,
        'phase': phase,
        'geocent_time': times,
        'a_1': a_1,
        'a_2': a_2,
        'tilt_1': tilt_1,
        'tilt_2': tilt_2,
        'phi_12': phi_12,
        'phi_jl': phi_jl}
df_binaries = pd.DataFrame(data)

In [ ]:
# save the dataframe
if hi_probability: 
    df_binaries.to_hdf('gw_data/bbh_1e4_events_for_gwfish_rs%s_nside%s_HI.hdf5'%(my_random_seed, nside), key='df', mode='w')
else:
    df_binaries.to_hdf('gw_data/bbh_1e4_events_for_gwfish_rs%s_nside%s_NoHI.hdf5'%(my_random_seed, nside), key='df', mode='w')